# Module 01 · LLM basics and the AI gateway

**How to use this notebook**
- Read the matching section of the chapter first, then run that section here.
- Run the cells **in order, top to bottom** (Shift + Enter). Each section builds on the one before.
- Every code cell has a note above it (*what this does*) and below it (*what you just saw*).
- Cost: a few cents in total. A spend cap of $1 per session stops anything running away.

| Section | Concept |
|---|---|
| Setup | Load settings, connect to the gateway |
| 1 | What an LLM call is |
| 2 | Why a gateway *(reading only)* |
| 3 | Meet the gateway: aliases |
| 4 | One client, two vendors |
| 5 | Tokens, cost and knobs |
| 6 | Spend caps |
| 7 | The model has no memory |
| 8 | The app: summarize a document |
| Break it | Four experiments |
| Checkpoint | Five questions |

## Setup

**What this cell does**
- **Colab:** downloads the course, installs DocuMind, and reads your two API keys from Colab Secrets (the key icon in the left sidebar).
- **Mac:** nothing to install here. You opened this notebook with `uv run --project app jupyter lab`, so DocuMind is already installed, and your keys are read from the repo's `.env` file.
- It also sets the app's log level to WARNING, so only the cells' own output shows.
- **`MOCK = True`** switches on practice mode: every call returns a canned reply, instantly and for free. Use it to learn the plumbing, then switch back to see real answers.

In [ ]:
MOCK = False   # True = free practice mode (canned replies, no vendor calls)

import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "https://github.com/panwars87/sha-genai-learning.git"
    BRANCH = "main"   # change if you are studying a different branch
    if not os.path.isdir("/content/sha-genai-learning"):
        !git clone -q -b {BRANCH} {REPO} /content/sha-genai-learning
    %cd /content/sha-genai-learning/modules/01-llm-basics-and-ai-gateway
    !pip -q install -e ./app
    from google.colab import userdata
    for name in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            print(f"Add the Colab Secret {name} (key icon, left sidebar), or set MOCK = True.")

# Keep the notebook tidy: show only warnings from the app's log (the CLI logs every call).
# This is a settings override by environment variable, the same way Colab or production would.
os.environ.setdefault("DOCUMIND_LOGGING__LEVEL", "WARNING")

if MOCK:
    os.environ["DOCUMIND_GATEWAY__MOCK_RESPONSE"] = "[mock reply] Switch MOCK to False for real answers."
print("Ready." + ("  (mock mode)" if MOCK else ""))

**Next:** load the settings and build the gateway. `settings.yaml` holds every knob, so nothing in the code below is hard-coded. The small `show()` helper prints an answer together with the facts the gateway reports about it.

In [ ]:
from documind.config import get_settings
from documind.llm import LLMError, build_gateway, chat

settings = get_settings()          # settings.yaml, overridden by any DOCUMIND_* env variables
gateway = build_gateway(settings)  # LiteLLM router: aliases → real models, retries, spend cap

# Which keys are present? (Names only. Values are never printed.)
if settings.gateway.mock_response is None:
    for name in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        print(f"{name:<18} {'found' if os.getenv(name) else 'MISSING: add it to .env'}")


def show(reply):
    """Print an answer and its metrics."""
    print(reply.text.strip() or "(empty answer)")
    print(f"\n  alias asked for : {reply.alias}")
    print(f"  model answered  : {reply.model}")
    print(f"  finish reason   : {reply.finish_reason}")
    print(f"  tokens in / out : {reply.input_tokens} / {reply.output_tokens}"
          + (f"  (thinking: {reply.reasoning_tokens})" if reply.reasoning_tokens else ""))
    print(f"  cost            : ${reply.cost_usd:.6f}")
    print(f"  latency         : {reply.latency_ms:,.0f} ms")

**What you just saw:** both keys should say `found`. If one says `MISSING`, add it to `.env` (Mac) or to Colab Secrets, then restart the kernel and run from the top.

---

## Section 1 · What an LLM call is

> **Explain like I'm 10:** an LLM is a super-autocomplete. It reads your words as small pieces called *tokens* and keeps guessing the next one until the answer is done. Then it forgets everything.

**What this cell does:** sends **one** request the "raw" way, straight to one vendor model by its full name. This is how most tutorials start. Watch the three parts of every answer:
1. the **text**,
2. the **usage** (tokens in and out, which is what you pay for),
3. the **finish reason** (`stop` = finished, `length` = cut off).

In [ ]:
import litellm

PROMPT = "In one sentence, what is an invoice?"

response = litellm.completion(
    model="anthropic/claude-haiku-5-5",           # a vendor model name, hard-wired: the old way
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=1000,
    mock_response=settings.gateway.mock_response,  # None unless MOCK is on
)

print("Text         :", response.choices[0].message.content)
print("Usage        :", response.usage.prompt_tokens, "in /", response.usage.completion_tokens, "out")
print("Finish reason:", response.choices[0].finish_reason)

**What you just saw**
- An LLM call is just **messages in, one message out**.
- The code names one vendor's model directly. To switch vendor you would edit code, and nothing here tracks or caps spend.
- Sections 3 to 6 fix that with a gateway.

---

## Section 2 · Why a gateway *(reading only)*

> **Explain like I'm 10:** a universal TV remote. One remote works every TV, and only the remote knows the passwords.

The chapter's diagram shows the idea. Every app talks to **one gateway**, and the gateway talks to the vendors. The gateway:

| Gives you | Instead of |
|---|---|
| **Aliases** (`fast`, `smart`) | Vendor model names scattered in code |
| One format for every vendor | One SDK per vendor |
| Cost and speed on every call | A surprise bill at month end |
| Spend caps | Nothing stopping a runaway loop |
| Retries (and later, fallbacks) | Rewriting resilience in every app |

In this course **LiteLLM runs as a library inside DocuMind**: nothing to install or run separately. In a company the same LiteLLM runs as a shared **server** in front of every team. The concepts are identical.

---

## Section 3 · Meet the gateway: aliases

**What this cell does:** prints the alias table from `app/config/settings.yaml`. These are the only model names DocuMind is allowed to use.

In [ ]:
print(f"{'alias':<17} → real model")
print("-" * 48)
for alias, model in settings.gateway.aliases.items():
    print(f"{alias:<17} → {model}")
print()
for short, target in settings.gateway.shortcuts.items():
    print(f"{short:<17} → {target}   (shortcut)")

**What you just saw:** four aliases, two per vendor, plus two shortcuts. The app says `fast`, and the gateway turns that into Claude Haiku.

**What this cell does:** the same question as Section 1, but through the gateway by alias. Compare the output: you now also get the real model's name, the **cost** and the **latency**.

In [ ]:
reply = chat(
    gateway,
    alias="fast",
    messages=[{"role": "user", "content": "In one sentence, what is an invoice?"}],
    max_output_tokens=settings.features.max_output_tokens,
)
show(reply)

**What you just saw:** you asked for `fast` and the gateway reports who actually answered and what it cost. Feature code never needs to know the vendor.

---

## Section 4 · One client, two vendors

**What this cell does:** sends the **same** prompt through the **same** function to Anthropic and to OpenAI. Only the alias string changes. Compare the wording, the token counts, the cost and the speed.

In [ ]:
PROMPT = "Explain in two sentences what a receipt proves, for a 10-year-old."

for alias in ("fast-anthropic", "fast-openai"):
    print(f"──── {alias} " + "─" * 40)
    try:
        show(chat(gateway, alias=alias, messages=[{"role": "user", "content": PROMPT}],
                  max_output_tokens=settings.features.max_output_tokens))
    except LLMError as error:
        print("✗", error)
    print()

**What you just saw:** vendor independence. Switching vendor is a **config** change, not a code change. In Module 04 the same mechanism gives automatic fallback when a vendor is down.

---

## Section 5 · Tokens, cost and knobs

### 5a · Tokens are the unit of cost
**What this cell does:** sends a tiny prompt, then a whole invoice, and shows how the character count turns into tokens and dollars. English averages about **4 characters per token**.

In [ ]:
from pathlib import Path

invoice_text = Path("../../data/invoices/INV-1001-acme-supplies.txt").read_text()

for label, prompt in [("tiny prompt", "Say hello."), ("whole invoice", "Summarize:\n" + invoice_text)]:
    r = chat(gateway, alias="fast", messages=[{"role": "user", "content": prompt}],
             max_output_tokens=settings.features.max_output_tokens)
    print(f"{label:<14} {len(prompt):>5} chars → {r.input_tokens:>4} tokens in, "
          f"{r.output_tokens:>4} out, ${r.cost_usd:.6f}")

**What you just saw:** cost grows with tokens, and **output tokens cost about 5× input tokens**. (Mock mode reports made-up token counts; set `MOCK = False` to see the real ones.) Current models also *think* before answering; thinking is billed as output.

### 5b · From one call to a monthly bill
**What this cell does:** takes the token counts of the invoice call above and prices them on both the fast and the smart model, for 10,000 invoices a month. This is the kind of calculation a Director signs off on.

In [ ]:
MONTHLY_INVOICES = 10_000

for alias in ("fast-anthropic", "smart-anthropic"):
    model = settings.gateway.aliases[alias]
    cost_in, cost_out = litellm.cost_per_token(
        model=model, prompt_tokens=r.input_tokens, completion_tokens=r.output_tokens
    )
    per_invoice = cost_in + cost_out
    print(f"{alias:<16} ${per_invoice:.5f} per invoice → ${per_invoice * MONTHLY_INVOICES:,.2f} per month")

**What you just saw:** the same job costs about **20× more** on the smart model. Picking the right model per task is *model routing* (Module 04), and checking quality before you switch is *evals* (Module 09).

### 5c · Same question, different answers
**What this cell does:** asks the same creative question three times.

In [ ]:
PROMPT = "Give me a creative one-line name for a café receipt scanner app."

for run in (1, 2, 3):
    r = chat(gateway, alias="fast", messages=[{"role": "user", "content": PROMPT}],
             max_output_tokens=settings.features.max_output_tokens)
    print(f"run {run}: {r.text.strip()}")

**What you just saw:** different answers to the same input (in mock mode they are all the same). LLMs are **not repeatable**, which is why "I tried it once and it worked" is not a quality bar.

### 5d · The old "temperature" knob
**What this cell does:** older tutorials set `temperature=0` to make answers more repeatable. Watch what a current Claude model says.

In [ ]:
try:
    r = chat(gateway, alias="fast", messages=[{"role": "user", "content": PROMPT}],
             max_output_tokens=settings.features.max_output_tokens, temperature=0.0)
    print("Accepted:", r.text.strip())
except LLMError as error:
    print("Rejected:", str(error).splitlines()[0])

**What you just saw:** current Claude models reject `temperature`. You get consistency from **clear prompts and structured output** (Module 02), and you prove it with **evals** (Module 09). Knobs from older tutorials can break on newer models.

---

## Section 6 · Spend caps

> **Explain like I'm 10:** pocket money. When it's spent, it's spent, however many times you ask.

In a company, the gateway gives each app a **virtual key** with its own budget (the chapter covers how). DocuMind does the same thing in miniature: a **spend tracker** checks the session's running cost **before** every call.

**What this cell does:** shows what this session has spent so far, then builds a second gateway with a tiny cap and watches it stop.

In [ ]:
print(f"This session: {gateway.spend.calls} calls, ${gateway.spend.spent_usd:.6f} "
      f"of a ${gateway.spend.limit_usd:.2f} cap\n")

tiny = build_gateway(settings)
tiny.spend.limit_usd = 0.000001   # far below the cost of one call

for attempt in (1, 2):
    try:
        chat(tiny, alias="fast", messages=[{"role": "user", "content": "Reply OK."}],
             max_output_tokens=settings.features.max_output_tokens)
        print(f"attempt {attempt}: ✓ allowed (spent ${tiny.spend.spent_usd:.6f})")
    except LLMError as error:
        print(f"attempt {attempt}: ✗ {error}")

**What you just saw:** the first call is allowed (nothing spent yet), and it pushes spend past the cap. The second is refused **before** any vendor is called, so a bug cannot run up a bill.

---

## Section 7 · The model has no memory

**What this cell does:** three requests.
1. Tell the model a fact.
2. Ask about it in a **new** request: it has no idea.
3. Ask again, **resending the conversation**: now it "remembers", and you pay for the extra tokens.

In [ ]:
FACT = "Remember this: invoice INV-1001 from Acme has a total of $1,101.60."
QUESTION = "What is the total of invoice INV-1001?"
limit = settings.features.max_output_tokens

first = chat(gateway, alias="fast", messages=[{"role": "user", "content": FACT}], max_output_tokens=limit)
second = chat(gateway, alias="fast", messages=[{"role": "user", "content": QUESTION}], max_output_tokens=limit)
history = [
    {"role": "user", "content": FACT},
    {"role": "assistant", "content": first.text},
    {"role": "user", "content": QUESTION},
]
third = chat(gateway, alias="fast", messages=history, max_output_tokens=limit)

print("2 · new request :", second.text.strip(), f"   [{second.input_tokens} tokens in]")
print("3 · with history:", third.text.strip(), f"   [{third.input_tokens} tokens in]")

**What you just saw:** chat apps "remember" only by **resending everything** on each turn. Long chats get slower and pricier, and eventually overflow the **context window**, the most a model can read at once. Choosing what to send is *context engineering* (Module 08).

---

## Section 8 · The app: summarize a document

**What this cell does:** runs DocuMind's real feature, the same code the command line uses. It loads and checks the file, fills the prompt templates from `app/prompts/`, and calls the gateway with the alias set in `settings.yaml`.

In [ ]:
from documind.summarize import summarize_document

for path in ("../../data/invoices/INV-1001-acme-supplies.txt",
             "../../data/receipts/RCPT-3001-bean-there-cafe.txt"):
    summary = summarize_document(Path(path), gateway)
    print(f"──── {summary.document_name}")
    show(summary.reply)
    print()

**What you just saw:** a working feature in about 30 lines (`app/src/documind/summarize.py`).

| File | Its one job |
|---|---|
| `config/settings.yaml` | Every setting: aliases, limits, spend cap |
| `prompts/*.md` | The prompt text, kept out of code |
| `documind/config.py` | Loads and validates settings |
| `documind/llm.py` | The gateway: aliases, cost, spend cap, clear errors |
| `documind/documents.py` | Checks the file before any money is spent |
| `documind/summarize.py` | The feature itself |

**What this cell does:** the same feature from the command line.

In [ ]:
!{sys.executable} -m documind summarize ../../data/receipts/RCPT-3002-handy-hardware.txt

---
## Break it

Four short experiments. Before running each one, predict what will happen.

### Experiment 1 · The answer gets cut off
**What this cell does:** allows only 16 output tokens for a question that needs a long answer. (In mock mode you'll see the canned reply instead.)

In [ ]:
r = chat(gateway, alias="fast",
         messages=[{"role": "user", "content": "List ten things every invoice should contain."}],
         max_output_tokens=16)
show(r)

**What you just saw:** `finish_reason` is `length`, and the answer is partial or even empty, because thinking used up the budget. A cut-off answer can look normal, so **always check finish_reason**. DocuMind logs a warning when it happens.

### Experiment 2 · Asking for a model the gateway doesn't allow
**What this cell does:** feature code asks for `gpt-4`, which isn't in `settings.yaml`.

In [ ]:
try:
    chat(gateway, alias="gpt-4", messages=[{"role": "user", "content": "hi"}], max_output_tokens=50)
except LLMError as error:
    print("✗", error)

**What you just saw:** the gateway refuses anything that isn't an approved alias. In a bank, this is how you enforce an **approved-model list**.

### Experiment 3 · Switch vendor with configuration only
**What this cell does:** points the `fast` shortcut at OpenAI by changing **settings only**, then runs the unchanged summarize feature.

In [ ]:
from documind.config import Settings

openai_settings = Settings(gateway={**settings.gateway.model_dump(),
                                    "shortcuts": {"fast": "fast-openai", "smart": "smart-openai"}})
summary = summarize_document(Path("../../data/invoices/INV-2047-globex-cloud.txt"),
                             build_gateway(openai_settings))
show(summary.reply)

**What you just saw:** the model changed to OpenAI and **no feature code changed**. You can do the same permanently by editing `shortcuts` in `settings.yaml`, or with the environment variable `DOCUMIND_GATEWAY__SHORTCUTS__FAST=fast-openai`.

### Experiment 4 · A wrong API key
**What this cell does:** makes one call with a deliberately fake key, to show what an authentication failure looks like. (Mock mode skips the vendor, so nothing fails there.)

In [ ]:
try:
    litellm.completion(model="anthropic/claude-haiku-5-5", api_key="not-a-real-key",
                       messages=[{"role": "user", "content": "hi"}], max_tokens=50,
                       mock_response=settings.gateway.mock_response)
    print("No error (mock mode never calls the vendor).")
except litellm.AuthenticationError as error:
    print("✗ AuthenticationError:", str(error)[:120], "…")

**What you just saw:** a clear `AuthenticationError`. In DocuMind, `llm.py` turns this into a message that tells you to check `.env`.

**Write what surprised you in `NOTES.md`.**

---
## Checkpoint

Answer out loud, then open each one.

<details><summary>1. A chat app "remembers" earlier messages. How, if the model is stateless?</summary>
It resends the whole conversation with every request, so memory costs tokens on every turn.
</details>

<details><summary>2. Name three things a gateway gives you that direct vendor calls don't.</summary>
Aliases (switch vendor by config), one format for all vendors, cost and latency per call, spend caps, retries and fallbacks, an approved-model list. Any three.
</details>

<details><summary>3. In a company, how is a spend cap usually enforced?</summary>
The shared gateway gives each app or team a virtual key with a budget, and checks it before calling the vendor.
</details>

<details><summary>4. The summary looks complete but finish_reason is "length". What do you do?</summary>
It was cut off, and thinking tokens count toward the limit. Raise <code>max_output_tokens</code> or ask for less, and always check finish_reason.
</details>

<details><summary>5. A team wants to cut cost 20× by moving 10,000 invoices a month to the fast model. What is your first question?</summary>
Does quality hold? Answering that needs evals (Module 09), not a single test.
</details>

In [ ]:
print(f"Session total: {gateway.spend.calls} calls, ${gateway.spend.spent_usd:.4f} spent "
      f"(cap ${gateway.spend.limit_usd:.2f}).")